In [14]:

import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from tabulate import tabulate
 
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
 

PERSIST_DIR = "../vectorstore"
COLLECTION_NAME = "nextjs"
EMBEDDING_MODEL = "text-embedding-3-small"
TOP_K = 5 
 

In [15]:
EVAL_DATASET = [
    {
        "query": "How do I create a dynamic route in Next.js?",
        "expected_keywords": ["dynamic", "params", "slug", "[slug]", "page"],
        "expected_sources": ["dynamic-routes", "routing"],
        "category": "Routing",
    },
    {
        "query": "What is a layout in the App Router?",
        "expected_keywords": ["layout", "shared", "children", "app"],
        "expected_sources": ["layout", "layouts"],
        "category": "Routing",
    },
    {
        "query": "How do route groups work in Next.js?",
        "expected_keywords": ["route group", "(folder)", "parentheses", "organize"],
        "expected_sources": ["route-groups"],
        "category": "Routing",
    },
    {
        "query": "What are parallel routes in Next.js?",
        "expected_keywords": ["parallel", "slot", "@", "simultaneously"],
        "expected_sources": ["parallel-routes"],
        "category": "Routing",
    },
    {
        "query": "How do intercepting routes work?",
        "expected_keywords": ["intercepting", "modal", "(..)", "(.)"],
        "expected_sources": ["intercepting-routes"],
        "category": "Routing",
    },
    {
        "query": "What is the loading.js file used for?",
        "expected_keywords": ["loading", "Suspense", "skeleton", "streaming"],
        "expected_sources": ["loading", "loading-ui"],
        "category": "Routing",
    },
    {
        "query": "How does the error.js boundary work?",
        "expected_keywords": ["error", "boundary", "reset", "Error"],
        "expected_sources": ["error", "error-handling"],
        "category": "Routing",
    },
    {
        "query": "How to use the Link component for navigation?",
        "expected_keywords": ["Link", "href", "navigation", "prefetch"],
        "expected_sources": ["link", "linking-and-navigating", "linking"],
        "category": "Routing",
    },
 
    # ── DATA FETCHING ──
    {
        "query": "How to fetch data in a Server Component?",
        "expected_keywords": ["server", "async", "fetch", "component"],
        "expected_sources": ["fetching-data", "data-fetching", "server-components"],
        "category": "Data Fetching",
    },
    {
        "query": "What is streaming and how does Suspense work in Next.js?",
        "expected_keywords": ["streaming", "Suspense", "fallback", "loading"],
        "expected_sources": ["streaming", "loading", "fetching"],
        "category": "Data Fetching",
    },
    {
        "query": "How to fetch data on the client side in Next.js?",
        "expected_keywords": ["client", "useEffect", "SWR", "React Query"],
        "expected_sources": ["fetching-data", "data-fetching", "client-components"],
        "category": "Data Fetching",
    },
    {
        "query": "How do Server Actions and mutations work?",
        "expected_keywords": ["server action", "use server", "form", "mutation"],
        "expected_sources": ["server-actions", "mutations", "form"],
        "category": "Data Fetching",
    },
 
    # ── CACHING ──
    {
        "query": "How does caching work in Next.js App Router?",
        "expected_keywords": ["cache", "revalidate", "static", "dynamic"],
        "expected_sources": ["caching"],
        "category": "Caching",
    },
    {
        "query": "What is the use cache directive?",
        "expected_keywords": ["use cache", "cache", "directive", "cacheLife"],
        "expected_sources": ["use-cache", "caching", "cache"],
        "category": "Caching",
    },
    {
        "query": "How to revalidate cached data on demand?",
        "expected_keywords": ["revalidateTag", "revalidatePath", "on-demand"],
        "expected_sources": ["caching", "revalidat"],
        "category": "Caching",
    },
    {
        "query": "What is Incremental Static Regeneration (ISR)?",
        "expected_keywords": ["ISR", "revalidate", "static", "regeneration", "incremental"],
        "expected_sources": ["incremental-static-regeneration", "caching", "isr"],
        "category": "Caching",
    },
 
    # ── RENDERING ──
    {
        "query": "What is the difference between Server and Client Components?",
        "expected_keywords": ["server component", "client component", "use client", "rendering"],
        "expected_sources": ["server-components", "client-components", "rendering"],
        "category": "Rendering",
    },
    {
        "query": "How does static rendering work in Next.js?",
        "expected_keywords": ["static", "build", "prerender", "SSG"],
        "expected_sources": ["static-rendering", "rendering", "static"],
        "category": "Rendering",
    },
    {
        "query": "What is Partial Prerendering (PPR)?",
        "expected_keywords": ["partial", "prerendering", "PPR", "shell", "streaming"],
        "expected_sources": ["partial-prerendering", "ppr", "rendering"],
        "category": "Rendering",
    },
 
    # ── MIDDLEWARE / PROXY ──
    {
        "query": "How to use middleware in Next.js?",
        "expected_keywords": ["middleware", "NextResponse", "matcher", "request"],
        "expected_sources": ["middleware", "proxy"],
        "category": "Middleware",
    },
    {
        "query": "How does the proxy.ts file replace middleware?",
        "expected_keywords": ["proxy", "middleware", "NextRequest", "runtime"],
        "expected_sources": ["proxy", "middleware"],
        "category": "Middleware",
    },
 
    # ── ROUTE HANDLERS / API ──
    {
        "query": "How to create an API route handler in the App Router?",
        "expected_keywords": ["route", "handler", "GET", "POST", "Response"],
        "expected_sources": ["route-handlers", "route"],
        "category": "API",
    },
    {
        "query": "How to handle different HTTP methods in route handlers?",
        "expected_keywords": ["GET", "POST", "PUT", "DELETE", "handler"],
        "expected_sources": ["route-handlers", "route"],
        "category": "API",
    },
 
    # ── STYLING ──
    {
        "query": "How to use CSS Modules in Next.js?",
        "expected_keywords": ["CSS Modules", "module.css", "className", "styles"],
        "expected_sources": ["css", "css-modules", "styling"],
        "category": "Styling",
    },
    {
        "query": "How to set up Tailwind CSS with Next.js?",
        "expected_keywords": ["Tailwind", "tailwind.config", "PostCSS"],
        "expected_sources": ["tailwind", "css", "styling"],
        "category": "Styling",
    },
 
    # ── OPTIMIZATION ──
    {
        "query": "How does the Next.js Image component optimize images?",
        "expected_keywords": ["Image", "next/image", "width", "height", "lazy", "optimization"],
        "expected_sources": ["image", "image-optimization", "optimizing"],
        "category": "Optimization",
    },
    {
        "query": "How to optimize fonts in Next.js?",
        "expected_keywords": ["font", "next/font", "Google", "preload"],
        "expected_sources": ["font", "fonts", "optimizing"],
        "category": "Optimization",
    },
    {
        "query": "How to add metadata and SEO tags in Next.js?",
        "expected_keywords": ["metadata", "title", "description", "generateMetadata", "SEO"],
        "expected_sources": ["metadata", "seo"],
        "category": "Optimization",
    },
    {
        "query": "How to configure the Next.js Script component?",
        "expected_keywords": ["Script", "next/script", "strategy", "lazyOnload"],
        "expected_sources": ["script", "optimizing"],
        "category": "Optimization",
    },
 
    # ── CONFIGURATION ──
    {
        "query": "How to configure next.config.js?",
        "expected_keywords": ["next.config", "NextConfig", "module.exports"],
        "expected_sources": ["turbopackFileSystemCache"],
        "category": "Configuration",
    },
    {
        "query": "How to set up environment variables in Next.js?",
        "expected_keywords": ["env", "NEXT_PUBLIC", ".env.local", "environment"],
        "expected_sources": ["environment-variables", "env"],
        "category": "Configuration",
    },
    {
        "query": "How to configure TypeScript in a Next.js project?",
        "expected_keywords": ["TypeScript", "tsconfig", "type checking"],
        "expected_sources": ["typescript"],
        "category": "Configuration",
    },
 
    # ── DEPLOYMENT ──
    {
        "query": "How to deploy a Next.js app?",
        "expected_keywords": ["deploy", "build", "start", "production", "Vercel"],
        "expected_sources": ["deploy", "deploying", "production"],
        "category": "Deployment",
    },
    {
        "query": "How does static export work in Next.js?",
        "expected_keywords": ["static", "export", "output", "HTML"],
        "expected_sources": ["static-export", "static-exports", "deploying"],
        "category": "Deployment",
    },
 
    # ── PROJECT STRUCTURE ──
    {
        "query": "What is the recommended project structure for Next.js?",
        "expected_keywords": ["app", "pages", "public", "src", "structure"],
        "expected_sources": ["index", "04-glossary"],
        "category": "Project Structure",
    },
    {
        "query": "What are the special files in the App Router?",
        "expected_keywords": ["page", "layout", "loading", "error", "not-found"],
        "expected_sources": ["index", "04-glossary"],
        "category": "Project Structure",
    },
 
    # ── AUTHENTICATION & SECURITY ──
    {
        "query": "How to implement authentication in Next.js?",
        "expected_keywords": ["auth", "session", "login", "protect"],
        "expected_sources": ["authentication", "auth"],
        "category": "Auth",
    },
 
    # ── INTERNATIONALIZATION ──
    {
        "query": "How to add internationalization (i18n) to a Next.js app?",
        "expected_keywords": ["i18n", "locale", "internationalization", "language"],
        "expected_sources": ["internationalization", "i18n"],
        "category": "i18n",
    },
 
    # ── TESTING ──
    {
        "query": "How to set up testing in Next.js with Jest?",
        "expected_keywords": ["Jest", "test", "testing", "config"],
        "expected_sources": ["jest", "testing"],
        "category": "Testing",
    },
    {
        "query": "How to use Playwright for end-to-end testing in Next.js?",
        "expected_keywords": ["Playwright", "e2e", "test", "browser"],
        "expected_sources": ["playwright", "testing"],
        "category": "Testing",
    },
 
    # ── INSTALLATION ──
    {
        "query": "How to install and create a new Next.js project?",
        "expected_keywords": ["create-next-app", "install", "npx", "npm"],
        "expected_sources": ["installation", "getting-started"],
        "category": "Getting Started",
    },
]

In [16]:
def keyword_match_score(retrieved_texts: list[str], expected_keywords: list[str]) -> dict:
    """
    Checks how many expected keywords appear in the retrieved chunks.
    Returns precision-style score: found_keywords / total_expected_keywords
    """
    combined_text = " ".join(retrieved_texts).lower()
    found = []
    missing = []
 
    for kw in expected_keywords:
        if kw.lower() in combined_text:
            found.append(kw)
        else:
            missing.append(kw)
 
    score = len(found) / len(expected_keywords) if expected_keywords else 0.0
    return {
        "score": score,
        "found": found,
        "missing": missing,
    }
 
 
def source_match_score(retrieved_sources: list[str], expected_sources: list[str]) -> dict:
    """
    Checks if ANY expected source filename (partial match) appears in retrieved results.
    Returns:
      - hit: bool (at least one expected source found)
      - rank: position of first match (1-indexed), or None
      - matched_source: the matched filename, or None
    """
    for rank, src in enumerate(retrieved_sources, start=1):
        src_lower = src.lower()
        for expected in expected_sources:
            if expected.lower() in src_lower:
                return {
                    "hit": True,
                    "rank": rank,
                    "matched_source": src,
                }
 
    return {"hit": False, "rank": None, "matched_source": None}
 
 
def reciprocal_rank(retrieved_sources: list[str], expected_sources: list[str]) -> float:
    """
    Reciprocal Rank: 1/position of first relevant result.
    Returns 0.0 if no relevant result found in top-k.
    """
    for rank, src in enumerate(retrieved_sources, start=1):
        src_lower = src.lower()
        for expected in expected_sources:
            if expected.lower() in src_lower:
                return 1.0 / rank
    return 0.0
 
 
def run_evaluation(
    vector_store: Chroma,
    dataset: list[dict],
    top_k: int = TOP_K,
) -> dict:
    """
    Run full evaluation over the dataset.
    Returns per-query results and aggregate metrics.
    """
    results = []
 
    for i, test_case in enumerate(dataset):
        query = test_case["query"]
        expected_kw = test_case["expected_keywords"]
        expected_src = test_case["expected_sources"]
        category = test_case["category"]
 
        # Retrieve
        docs = vector_store.similarity_search(query, k=top_k)
 
        retrieved_texts = [doc.page_content for doc in docs]
        retrieved_sources = [doc.metadata.get("source", "") for doc in docs]
 
        # Evaluate
        kw_result = keyword_match_score(retrieved_texts, expected_kw)
        src_result = source_match_score(retrieved_sources, expected_src)
        rr = reciprocal_rank(retrieved_sources, expected_src)
 
        results.append({
            "id": i + 1,
            "query": query,
            "category": category,
            "keyword_score": kw_result["score"],
            "keywords_found": kw_result["found"],
            "keywords_missing": kw_result["missing"],
            "source_hit": src_result["hit"],
            "source_rank": src_result["rank"],
            "matched_source": src_result["matched_source"],
            "reciprocal_rank": rr,
            "retrieved_sources": retrieved_sources,
        })
 
    # ── Aggregate Metrics ──
    n = len(results)
    avg_keyword = sum(r["keyword_score"] for r in results) / n
    source_hit_rate = sum(1 for r in results if r["source_hit"]) / n
    mrr = sum(r["reciprocal_rank"] for r in results) / n
 
    # ── Per-Category Metrics ──
    categories = {}
    for r in results:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = {"keyword_scores": [], "source_hits": [], "rrs": []}
        categories[cat]["keyword_scores"].append(r["keyword_score"])
        categories[cat]["source_hits"].append(r["source_hit"])
        categories[cat]["rrs"].append(r["reciprocal_rank"])
 
    category_metrics = {}
    for cat, vals in categories.items():
        n_cat = len(vals["keyword_scores"])
        category_metrics[cat] = {
            "count": n_cat,
            "avg_keyword_score": sum(vals["keyword_scores"]) / n_cat,
            "source_hit_rate": sum(vals["source_hits"]) / n_cat,
            "mrr": sum(vals["rrs"]) / n_cat,
        }
 
    return {
        "per_query": results,
        "aggregate": {
            "total_queries": n,
            "avg_keyword_match": round(avg_keyword, 4),
            "source_hit_rate": round(source_hit_rate, 4),
            "mrr": round(mrr, 4),
        },
        "per_category": category_metrics,
    }
 

In [17]:
def print_results(eval_results: dict):
    """Pretty-print evaluation results."""
 
    agg = eval_results["aggregate"]
 
    # ── Header ──
    print("\n" + "=" * 70)
    print("  RETRIEVAL EVALUATION RESULTS")
    print("=" * 70)
 
    # ── Aggregate Metrics ──
    print(f"\n  Total Queries:       {agg['total_queries']}")
    print(f"  Avg Keyword Match:   {agg['avg_keyword_match']:.2%}")
    print(f"  Source Hit Rate:     {agg['source_hit_rate']:.2%}")
    print(f"  MRR:                 {agg['mrr']:.4f}")
 
    # ── Per-Category Table ──
    print(f"\n{'─' * 70}")
    print("  PER-CATEGORY BREAKDOWN")
    print(f"{'─' * 70}")
 
    cat_table = []
    for cat, m in sorted(eval_results["per_category"].items()):
        cat_table.append([
            cat,
            m["count"],
            f"{m['avg_keyword_score']:.2%}",
            f"{m['source_hit_rate']:.2%}",
            f"{m['mrr']:.4f}",
        ])
 
    print(tabulate(
        cat_table,
        headers=["Category", "Queries", "Keyword Match", "Source Hit", "MRR"],
        tablefmt="simple_outline",
    ))
 
    # ── Per-Query Details ──
    print(f"\n{'─' * 70}")
    print("  PER-QUERY DETAILS")
    print(f"{'─' * 70}")
 
    query_table = []
    for r in eval_results["per_query"]:
        status = "✓" if r["source_hit"] else "✗"
        query_short = r["query"][:50] + "..." if len(r["query"]) > 50 else r["query"]
        query_table.append([
            r["id"],
            query_short,
            f"{r['keyword_score']:.0%}",
            status,
            r["source_rank"] or "-",
            f"{r['reciprocal_rank']:.2f}",
        ])
 
    print(tabulate(
        query_table,
        headers=["#", "Query", "KW Match", "Src", "Rank", "RR"],
        tablefmt="simple_outline",
    ))
 
    # ── Failed Queries (no source match) ──
    failures = [r for r in eval_results["per_query"] if not r["source_hit"]]
    if failures:
        print(f"\n{'─' * 70}")
        print(f"  FAILED SOURCE MATCHES ({len(failures)} queries)")
        print(f"{'─' * 70}")
        for r in failures:
            print(f"\n  [{r['id']}] {r['query']}")
            print(f"      Expected:  {', '.join(EVAL_DATASET[r['id']-1]['expected_sources'])}")
            print(f"      Got:       {', '.join(r['retrieved_sources'])}")
            print(f"      Missing KW: {', '.join(r['keywords_missing'])}")
 
    print(f"\n{'=' * 70}\n")
 
 
def list_sources(vector_store: Chroma, sample_size: int = 20):
    """
    Utility: list unique source filenames in the vector store.
    Run this first to calibrate expected_sources in your dataset.
    """
    results = vector_store.get(limit=sample_size, include=["metadatas"])
    sources = set()
    for meta in results["metadatas"]:
        if meta and "source" in meta:
            sources.add(meta["source"])
 
    print("\n  SOURCE FILES IN VECTOR STORE:")
    print("  " + "-" * 40)
    for s in sorted(sources):
        print(f"    {s}")
    print()
    return sorted(sources)
 
 
def save_results(eval_results: dict, output_path: str = "eval_results.json"):
    """Save full evaluation results to JSON for further analysis."""
 
    # Make JSON-serializable
    serializable = {
        "aggregate": eval_results["aggregate"],
        "per_category": eval_results["per_category"],
        "per_query": eval_results["per_query"],
    }
 
    with open(output_path, "w") as f:
        json.dump(serializable, f, indent=2)
    print(f"  Results saved to: {output_path}")
 
 
 
    

In [18]:

print("\n  Loading vector store...")
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)

# Step 0: List actual sources (run once to calibrate expected_sources)
print("\n  Step 0: Listing sources in your vector store")
all_sources = list_sources(vector_store, sample_size=500)

# Step 1: Run evaluation
print("  Step 1: Running retrieval evaluation...")
start = time.time()
eval_results = run_evaluation(vector_store, EVAL_DATASET, top_k=TOP_K)
elapsed = time.time() - start
print(f"  Done in {elapsed:.1f}s ({len(EVAL_DATASET)} queries, top_k={TOP_K})")

# Step 2: Display results
print_results(eval_results)

# Step 3: Save to JSON
save_results(eval_results)


  Loading vector store...

  Step 0: Listing sources in your vector store

  SOURCE FILES IN VECTOR STORE:
  ----------------------------------------
    01-installation.mdx
    02-project-structure.mdx
    03-layouts-and-pages.mdx
    04-linking-and-navigating.mdx
    05-server-and-client-components.mdx
    06-fetching-data.mdx
    07-mutating-data.mdx
    08-caching.mdx
    09-revalidating.mdx
    10-error-handling.mdx
    11-css.mdx
    12-images.mdx
    13-fonts.mdx
    14-metadata-and-og-images.mdx
    15-route-handlers.mdx
    16-proxy.mdx
    17-deploying.mdx
    18-upgrading.mdx
    ai-agents.mdx
    analytics.mdx
    authentication.mdx
    backend-for-frontend.mdx
    caching-without-cache-components.mdx
    cdn-caching.mdx
    ci-build-caching.mdx
    content-security-policy.mdx
    index.mdx

  Step 1: Running retrieval evaluation...
  Done in 14.7s (41 queries, top_k=5)

  RETRIEVAL EVALUATION RESULTS

  Total Queries:       41
  Avg Keyword Match:   88.29%
  Source Hit Ra